In [1]:
import os
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score)
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBClassifier
import joblib
from tqdm import tqdm

warnings.filterwarnings("ignore")


In [2]:
df_cleaned = pd.read_csv('../data/processed/clean_customer_dataNEW.csv')
df_cleaned

,customer_id,residence_country,gender,age,first_join_date,residence_index,channel_entrace,activity_status,household_gross_income,saving_account,...,avg_expense_days_per_month,expense_amount_cv,avg_transactions_per_month,monthly_transaction_std,total_transactions,active_months,avg_monthly_transaction_count,SPS,TSI,demographic_score
0,1375586,ES,1,35,2020-01-12,Y,KHL,1,50887.44,1,...,1.272727,0.939871,1.250000,0.452267,15.0,12.0,1.25,0.122553,0.485454,0.381798
1,1050611,ES,0,23,2017-08-10,Y,KHE,1,30619.38,1,...,1.285714,1.348381,1.357143,0.633324,19.0,14.0,1.36,0.090260,0.201175,0.168332
2,1050612,ES,0,23,2017-08-10,Y,KHE,1,57420.17,0,...,1.000000,0.732798,1.000000,0.000000,10.0,10.0,1.00,0.142588,0.633601,0.418683
3,1050613,ES,0,22,2017-08-10,Y,KHD,1,115661.59,0,...,1.266667,0.709893,1.312500,0.602080,21.0,16.0,1.31,0.103972,0.541213,0.428340
4,1050614,ES,0,23,2017-08-10,Y,KHE,1,28358.36,0,...,1.125000,0.871065,1.125000,0.353553,9.0,8.0,1.12,0.146832,0.539062,0.417684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939465,1185013,ES,0,53,2021-05-14,Y,KDB,1,116170.73,1,...,1.100000,1.455297,1.090909,0.301511,12.0,11.0,1.09,0.161168,0.223578,0.454709
939466,1168909,ES,1,43,2018-08-23,Y,KDB,1,5589.71,1,...,1.000000,1.409169,1.000000,0.000000,10.0,10.0,1.00,0.140636,0.295416,0.225875
939467,1173729,ES,1,33,2018-09-09,Y,KDB,1,19151.20,0,...,1.111111,0.724515,1.200000,0.421637,12.0,10.0,1.20,0.180721,0.579566,0.197636
939468,1164094,ES,0,54,2021-05-13,Y,KFC,0,13525.97,0,...,1.333333,1.033329,1.333333,1.154701,16.0,12.0,1.33,0.099282,0.639170,0.436374


In [3]:
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 939470 entries, 0 to 939469
Data columns (total 35 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   customer_id                    939470 non-null  int64  
 1   residence_country              939470 non-null  str    
 2   gender                         939470 non-null  int64  
 3   age                            939470 non-null  int64  
 4   first_join_date                939470 non-null  str    
 5   residence_index                939470 non-null  str    
 6   channel_entrace                939470 non-null  str    
 7   activity_status                939470 non-null  int64  
 8   household_gross_income         939470 non-null  float64
 9   saving_account                 939470 non-null  int64  
 10  guarantees                     939470 non-null  int64  
 11  junior_account                 939470 non-null  int64  
 12  loans                          939470 non

In [4]:

# Define label columns
label_cols = [
    'saving_account', 'guarantees', 'junior_account', 'loans',
     'pension'
]

# Prepare features and labels
X_raw = df_cleaned.drop(columns=label_cols)
drop_cols = ['credit_card','direct_debit','customer_id', 'first_join_date', 'total_transactions',
             'avg_monthly_transaction_count', 'demographic_score', 'customer_segment','min_balance','max_balance','avg_balance','current_loan_amount','credit_score']
X = X_raw.drop(columns=[col for col in drop_cols if col in X_raw.columns])
Y = df_cleaned[label_cols]

# Convert categorical columns
cat_cols = ['residence_country', 'residence_index', 'channel_entrace']
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].astype('category')
for col in X.select_dtypes(include='object').columns:
    X[col] = X[col].astype('category')

# Derive membership_days from first_join_date
if 'first_join_date' in df_cleaned.columns:
    join_date = pd.to_datetime(df_cleaned['first_join_date'], errors='coerce')
    X['membership_days'] = (pd.Timestamp.now() - join_date).dt.days

In [5]:
# Train XGBoost and LightGBM Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import time

# Split into training and test sets first
X_train_full, X_test, Y_train_full, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# Set up parameter grids
xgb_param_grid = {
    'estimator__n_estimators': [100],
    'estimator__max_depth': [3, 5],
    'estimator__learning_rate': [0.1],
    'estimator__subsample': [1.0]
}

lgb_param_grid = {
    'estimator__n_estimators': [100],
    'estimator__max_depth': [3, 5],
    'estimator__learning_rate': [0.1],
    'estimator__subsample': [1.0]
}

# XGBoost Training
print("Training XGBoost MultiOutputClassifier with GridSearch...")
base_xgb = XGBClassifier(
    tree_method='hist',
    eval_metric='logloss',
    use_label_encoder=False,
    enable_categorical=True,
    random_state=42
)
multi_xgb = MultiOutputClassifier(base_xgb, n_jobs=-1)
grid_xgb = GridSearchCV(
    estimator=multi_xgb,
    param_grid=xgb_param_grid,
    scoring='f1_macro',
    cv=3,
    verbose=1,
    n_jobs=-1
)
start_time = time.time()
grid_xgb.fit(X_train_full, Y_train_full)
xgb_model = grid_xgb.best_estimator_
print(f"XGBoost training completed in {time.time() - start_time:.2f} seconds.")
print("Best XGBoost Parameters:", grid_xgb.best_params_)

# LightGBM Training
print("\nTraining LightGBM MultiOutputClassifier with GridSearch...")
base_lgb = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
multi_lgb = MultiOutputClassifier(base_lgb, n_jobs=-1)
grid_lgb = GridSearchCV(
    estimator=multi_lgb,
    param_grid=lgb_param_grid,
    scoring='f1_macro',
    cv=3,
    verbose=1,
    n_jobs=-1
)
start_time = time.time()
grid_lgb.fit(X_train_full, Y_train_full)
lgb_model = grid_lgb.best_estimator_
print(f"LightGBM training completed in {time.time() - start_time:.2f} seconds.")
print("Best LightGBM Parameters:", grid_lgb.best_params_)


Training XGBoost MultiOutputClassifier with GridSearch...
Fitting 3 folds for each of 2 candidates, totalling 6 fits
XGBoost training completed in 51.64 seconds.
Best XGBoost Parameters: {'estimator__learning_rate': 0.1, 'estimator__max_depth': 3, 'estimator__n_estimators': 100, 'estimator__subsample': 1.0}

Training LightGBM MultiOutputClassifier with GridSearch...
Fitting 3 folds for each of 2 candidates, totalling 6 fits
LightGBM training completed in 44.26 seconds.
Best LightGBM Parameters: {'estimator__learning_rate': 0.1, 'estimator__max_depth': 5, 'estimator__n_estimators': 100, 'estimator__subsample': 1.0}


In [6]:
# Check Overfitting for both models
def check_overfitting(model, model_name, X_train, Y_train, X_test, Y_test, label_cols):
    Y_train_pred = model.predict(X_train)
    Y_test_pred = model.predict(X_test)
    
    Y_train_proba = model.predict_proba(X_train)
    Y_test_proba = model.predict_proba(X_test)
    
    print(f"=== CHECK OVERFITTING: {model_name} ===\n")
    
    for i, label in enumerate(label_cols):
        y_train_true = Y_train[label].values
        y_test_true = Y_test[label].values
        
        y_train_p = Y_train_pred[:, i]
        y_test_p = Y_test_pred[:, i]
        
        y_train_prob = Y_train_proba[i][:, 1]
        y_test_prob = Y_test_proba[i][:, 1]
        
        train_f1 = f1_score(y_train_true, y_train_p, zero_division=0)
        test_f1 = f1_score(y_test_true, y_test_p, zero_division=0)
        
        train_auc = roc_auc_score(y_train_true, y_train_prob) if len(set(y_train_true)) > 1 else 1.0
        test_auc = roc_auc_score(y_test_true, y_test_prob) if len(set(y_test_true)) > 1 else 1.0
        
        diff_f1 = train_f1 - test_f1
        
        print(f"Product: {label}")
        print(f"  F1-Score  -> Train: {train_f1:.4f} | Test: {test_f1:.4f} (Selisih: {diff_f1:.4f})")
        print(f"  ROC AUC   -> Train: {train_auc:.4f} | Test: {test_auc:.4f}")
        if diff_f1 > 0.15:
            print("  ⚠️ Indikasi Overfitting (Selisih F1 > 0.15)")
        else:
            print("  ✅ Aman / Generalisasi Baik")
        print("-" * 50)
        
    macro_train = f1_score(Y_train, Y_train_pred, average='macro', zero_division=0)
    macro_test = f1_score(Y_test, Y_test_pred, average='macro', zero_division=0)
    print(f"\nOVERALL MACRO F1 SCORE FOR {model_name}:")
    print(f"Train Macro F1: {macro_train:.4f}")
    print(f"Test Macro F1: {macro_test:.4f}")
    print(f"Selisih Overall: {macro_train - macro_test:.4f}\n")
    return macro_train, macro_test

# Evaluate both models
_ = check_overfitting(xgb_model, "XGBoost", X_train_full, Y_train_full, X_test, Y_test, label_cols)
_ = check_overfitting(lgb_model, "LightGBM", X_train_full, Y_train_full, X_test, Y_test, label_cols)


=== CHECK OVERFITTING: XGBoost ===

Product: saving_account
  F1-Score  -> Train: 0.0093 | Test: 0.0060 (Selisih: 0.0033)
  ROC AUC   -> Train: 0.5174 | Test: 0.4995
  ✅ Aman / Generalisasi Baik
--------------------------------------------------
Product: guarantees
  F1-Score  -> Train: 0.0000 | Test: 0.0000 (Selisih: 0.0000)
  ROC AUC   -> Train: 0.9995 | Test: 0.7555
  ✅ Aman / Generalisasi Baik
--------------------------------------------------
Product: junior_account
  F1-Score  -> Train: 0.8873 | Test: 0.7849 (Selisih: 0.1024)
  ROC AUC   -> Train: 0.9999 | Test: 0.9995
  ✅ Aman / Generalisasi Baik
--------------------------------------------------
Product: loans
  F1-Score  -> Train: 0.7507 | Test: 0.7507 (Selisih: 0.0000)
  ROC AUC   -> Train: 0.5171 | Test: 0.4974
  ✅ Aman / Generalisasi Baik
--------------------------------------------------
Product: pension
  F1-Score  -> Train: 0.0000 | Test: 0.0000 (Selisih: 0.0000)
  ROC AUC   -> Train: 0.8000 | Test: 0.8017
  ✅ Aman / Gen

In [7]:
# Model Evaluate using thresholds
custom_thresholds = {
    'saving_account': 0.3,
    'guarantees': 0.5,
    'junior_account': 0.17,
    'loans': 0.5,
    'pension': 0.12,
}


In [8]:
def predict_with_custom_threshold(model, X_input, label_cols, thresholds):
    """
    Apply per-label thresholds on predicted probabilities from MultiOutputClassifier.
    Returns binary predictions.
    """
    probas = model.predict_proba(X_input)
    preds = []

    for i, prob in enumerate(probas):
        threshold = thresholds[label_cols[i]]
        pred_label = (prob[:, 1] >= threshold).astype(int)
        preds.append(pred_label)

    return np.column_stack(preds)


In [9]:
# Predict and Evaluate Custom Thresholds for both models
from sklearn.metrics import classification_report

Y_test_pred_custom_xgb = predict_with_custom_threshold(xgb_model, X_test, label_cols, custom_thresholds)
Y_test_pred_custom_lgb = predict_with_custom_threshold(lgb_model, X_test, label_cols, custom_thresholds)

print("=== XGBoost Classification Report (Custom Thresholds) ===")
print(classification_report(Y_test, Y_test_pred_custom_xgb, target_names=label_cols, zero_division=0))

print("\n=== LightGBM Classification Report (Custom Thresholds) ===")
print(classification_report(Y_test, Y_test_pred_custom_lgb, target_names=label_cols, zero_division=0))

custom_f1_xgb = f1_score(Y_test, Y_test_pred_custom_xgb, average='macro', zero_division=0)
custom_f1_lgb = f1_score(Y_test, Y_test_pred_custom_lgb, average='macro', zero_division=0)
print(f"XGBoost Test Macro F1 with Custom Thresholds: {custom_f1_xgb:.4f}")
print(f"LightGBM Test Macro F1 with Custom Thresholds: {custom_f1_lgb:.4f}")


=== XGBoost Classification Report (Custom Thresholds) ===
                precision    recall  f1-score   support

saving_account       0.45      1.00      0.62     84649
    guarantees       0.00      0.00      0.00         6
junior_account       0.71      0.96      0.82       198
         loans       0.60      1.00      0.75    112974
       pension       0.16      0.29      0.21      8873

     micro avg       0.51      0.97      0.67    206700
     macro avg       0.39      0.65      0.48    206700
  weighted avg       0.52      0.97      0.67    206700
   samples avg       0.52      0.77      0.60    206700


=== LightGBM Classification Report (Custom Thresholds) ===
                precision    recall  f1-score   support

saving_account       0.45      1.00      0.62     84649
    guarantees       0.00      0.00      0.00         6
junior_account       0.49      0.96      0.65       198
         loans       0.60      0.49      0.54    112974
       pension       0.07      0.98   

In [10]:
# Detailed Evaluation (Accuracy, AUC) for both models on Test Set
def print_detailed_evaluation(model, model_name, X_test, Y_test, label_cols):
    print(f"=== Detailed Evaluation for {model_name} ===")
    Y_pred = model.predict(X_test)
    Y_pred_proba = model.predict_proba(X_test)
    for i, label in enumerate(label_cols):
        y_true = Y_test[label].values
        y_pred_col = Y_pred[:, i]
        y_proba = Y_pred_proba[i][:, 1]

        acc = accuracy_score(y_true, y_pred_col)
        auc = roc_auc_score(y_true, y_proba) if len(set(y_true)) > 1 else 1.0

        print(f"   Product: {label}")
        print(f"   Accuracy: {acc:.4f}")
        print(f"   AUC: {auc:.4f}\n")

print_detailed_evaluation(xgb_model, "XGBoost", X_test, Y_test, label_cols)
print_detailed_evaluation(lgb_model, "LightGBM", X_test, Y_test, label_cols)


=== Detailed Evaluation for XGBoost ===
   Product: saving_account
   Accuracy: 0.5491
   AUC: 0.4995

   Product: guarantees
   Accuracy: 1.0000
   AUC: 0.7555

   Product: junior_account
   Accuracy: 0.9995
   AUC: 0.9995

   Product: loans
   Accuracy: 0.6010
   AUC: 0.4974

   Product: pension
   Accuracy: 0.9528
   AUC: 0.8017

=== Detailed Evaluation for LightGBM ===
   Product: saving_account
   Accuracy: 0.4940
   AUC: 0.4998

   Product: guarantees
   Accuracy: 0.9998
   AUC: 0.6165

   Product: junior_account
   Accuracy: 0.9992
   AUC: 0.9842

   Product: loans
   Accuracy: 0.4965
   AUC: 0.4973

   Product: pension
   Accuracy: 0.6343
   AUC: 0.8043



In [11]:
from sklearn.metrics import mean_squared_error

# Detailed Evaluation (Accuracy, AUC) for both models on Test Set
def print_detailed_evaluation_RMSE(model, model_name, X_test, Y_test, label_cols):
    print(f"=== Detailed Evaluation for {model_name} ===")
    Y_pred = model.predict(X_test)
    for i, label in enumerate(label_cols):
        y_true = Y_test[label].values
        y_pred_col = Y_pred[:, i]
        rmse = np.sqrt(mean_squared_error(y_true, y_pred_col))

        print(f"   Product: {label}")
        print(f"   RMSE: {rmse:.4f}")

print_detailed_evaluation_RMSE(xgb_model, "XGBoost", X_test, Y_test, label_cols)

=== Detailed Evaluation for XGBoost ===
   Product: saving_account
   RMSE: 0.6715
   Product: guarantees
   RMSE: 0.0057
   Product: junior_account
   RMSE: 0.0220
   Product: loans
   RMSE: 0.6317
   Product: pension
   RMSE: 0.2173


In [12]:
from sklearn.metrics import mean_squared_error

# Detailed Evaluation (Accuracy, AUC) for both models on Test Set
def print_detailed_evaluation_RMSE(model, model_name, X_test, Y_test, label_cols):
    print(f"=== Detailed Evaluation for {model_name} ===")
    Y_pred = model.predict(X_test)
    for i, label in enumerate(label_cols):
        y_true = Y_test[label].values
        y_pred_col = Y_pred[:, i]
        rmse = np.sqrt(mean_squared_error(y_true, y_pred_col))

        print(f"   Product: {label}")
        print(f"   RMSE: {rmse:.4f}")

print_detailed_evaluation_RMSE(lgb_model, "LightGBM", X_test, Y_test, label_cols)

=== Detailed Evaluation for LightGBM ===
   Product: saving_account
   RMSE: 0.7113
   Product: guarantees
   RMSE: 0.0150
   Product: junior_account
   RMSE: 0.0276
   Product: loans
   RMSE: 0.7096
   Product: pension
   RMSE: 0.6047


In [13]:
y_pred_proba = xgb_model.predict_proba(X_test)
# print(Y_test.shape)
# print(y_pred_proba)

# Ambil probabilitas kelas positif dari setiap label
y_pred_proba_fixed = np.column_stack([
    pred[:, 1] for pred in y_pred_proba
])

roc_auc = roc_auc_score(
    Y_test,
    y_pred_proba_fixed,
    average='macro'
)

print("Macro ROC AUC:", roc_auc)

Macro ROC AUC: 0.7107124733012542


In [ ]:
# Compare Custom Macro F1 and save the final model
if custom_f1_lgb > custom_f1_xgb:
    final_model = lgb_model
    selected_name = "LightGBM"
else:
    final_model = xgb_model
    selected_name = "XGBoost"

print(f"Selected Model for Deployment: {selected_name} (Macro F1: {max(custom_f1_xgb, custom_f1_lgb):.4f})")

# Save final model
joblib.dump(final_model, '../models/best_multilabel_model.pkl')
print("Best model saved as best_multilabel_model.pkl")


Selected Model for Deployment: XGBoost (Macro F1: 0.4797)


In [15]:
# Define the recommend function
def recommend_top_n_products_filtered(model, X_input, Y_current, label_cols, top_n=3):
    probas = model.predict_proba(X_input)
    probs_matrix = np.column_stack([p[:, 1] for p in probas])

    probs_df = pd.DataFrame(probs_matrix, columns=label_cols, index=Y_current.index)

    masked_probs_df = probs_df.mask(Y_current == 1, -1)

    recommendations = []
    for _, customer_probs in masked_probs_df.iterrows():
        top_products = customer_probs.sort_values(ascending=False).head(top_n).index.tolist()
        recommendations.append(top_products)

    result_df = pd.DataFrame({
        'customer_index': Y_current.index,
        'recommended_products': recommendations
    })

    return result_df


In [ ]:
# Generate Recommendations
X_all = df_cleaned.drop(columns=label_cols)
Y_all = df_cleaned[label_cols]
customer_ids_all = df_cleaned['customer_id'].values

recommendations_all = recommend_top_n_products_filtered(
    model=final_model,
    X_input=X,
    Y_current=Y,
    label_cols=label_cols,
    top_n=3
)

recommendations_all['customer_id'] = customer_ids_all
recommendations_all = recommendations_all.drop(columns='customer_index')
recommendations_all = recommendations_all[['customer_id', 'recommended_products']]

# Save the recommendation csv
recommendations_all.to_csv('../data/processed/customer_data_recommendations.csv', index=False)
print("Recommendations saved to customer_data_recommendations.csv")


In [17]:
# Display first 5 recommendations
recommendations_all.head()


,customer_id,recommended_products
0,1375586,"[loans, pension, junior_account]"
1,1050611,"[loans, pension, junior_account]"
2,1050612,"[saving_account, pension, junior_account]"
3,1050613,"[saving_account, pension, junior_account]"
4,1050614,"[saving_account, pension, junior_account]"
